In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from urllib.parse import urljoin, urlparse
import PyPDF2
import io
from typing import List, Dict, Optional, Tuple
import logging
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Logging ayarları
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class TreasuryAuctionScraper:
    def __init__(self, max_pages: int = 50, headless: bool = True, analyze_strategy: bool = True):
        """
        Türkiye Hazinesi ihale sonuçları çekici
        
        Args:
            max_pages: Kontrol edilecek maksimum sayfa sayısı
            headless: Tarayıcıyı görünmez modda çalıştır
            analyze_strategy: İç Borçlanma Stratejisi analizi yapılsın mı
        """
        self.base_url = "https://www.hmb.gov.tr"
        self.category_url = "https://www.hmb.gov.tr/kategori/kamu-finansmani/sayfa/"
        self.max_pages = max_pages
        self.headless = headless
        self.analyze_strategy = analyze_strategy
        
        # Selenium driver ayarları
        self.chrome_options = Options()
        if headless:
            self.chrome_options.add_argument("--headless")
        self.chrome_options.add_argument("--no-sandbox")
        self.chrome_options.add_argument("--disable-dev-shm-usage")
        self.chrome_options.add_argument("--disable-gpu")
        self.chrome_options.add_argument("--window-size=1920,1080")
        self.chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        
        self.driver = None
        
        # Requests session
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        
        # Çekilecek alanlar
        self.fields = [
            'ISIN', 'Senet Tanımı', 'İhale Tarihi', 'Valör Tarihi', 'İtfa Tarihi', 'Vade (Yıl)',
            'ROT Toplam(Teklif)', 'ROT Toplam(Gerçekleşme)',
            'ROT Kamu(Teklif)', 'ROT Kamu(Gerçekleşme)',
            'ROT Piyasa Yapıcılar(Teklif)', 'ROT Piyasa Yapıcılar(Gerçekleşme)',
            'ROT Piyasa Yapıcılar Kabul Oranı (%)',
            'İhale(Teklif)', 'İhale(Gerçekleşme)',
            'İhale Kabul Oranı (%)',
            'Toplam(Teklif)', 'Toplam(Gerçekleşme)',
            'Ortalama Yıllık Basit(Teklif)', 'Ortalama Yıllık Basit(Gerçekleşme)',
            'Ortalama Yıllık Bileşik(Teklif)', 'Ortalama Yıllık Bileşik(Gerçekleşme)',
            'En Düşük Yıllık Bileşik(Teklif)', 'En Düşük Yıllık Bileşik(Gerçekleşme)',
            'En Yüksek Yıllık Bileşik(Teklif)', 'En Yüksek Yıllık Bileşik(Gerçekleşme)',
            'Ortalama Fiyat(Teklif)', 'Ortalama Fiyat(Gerçekleşme)',
            'En Düşük Fiyat(Teklif)', 'En Düşük Fiyat(Gerçekleşme)',
            'En Yüksek Fiyat(Teklif)', 'En Yüksek Fiyat(Gerçekleşme)'
        ]
    
    def init_driver(self):
        """Selenium driver'ı başlat"""
        try:
            self.driver = webdriver.Chrome(options=self.chrome_options)
            logger.info("Chrome driver başlatıldı")
        except Exception as e:
            logger.error(f"Chrome driver başlatılamadı: {str(e)}")
            raise
    
    def close_driver(self):
        """Selenium driver'ı kapat"""
        if self.driver:
            self.driver.quit()
            logger.info("Chrome driver kapatıldı")
    
    def get_auction_announcement_urls(self) -> List[str]:
        """
        Sayfalarda ihale sonuçları duyuru URL'lerini bulur
        """
        if not self.driver:
            self.init_driver()
            
        auction_urls = []
        
        for page in range(1, self.max_pages + 1):
            logger.info(f"Sayfa {page} kontrol ediliyor...")
            
            try:
                url = f"{self.category_url}{page}"
                self.driver.get(url)
                
                WebDriverWait(self.driver, 15).until(
                    EC.presence_of_element_located((By.TAG_NAME, "body"))
                )
                
                time.sleep(3)
                
                soup = BeautifulSoup(self.driver.page_source, 'html.parser')
                
                links = soup.find_all('a', href=True)
                logger.info(f"Sayfa {page}'de {len(links)} link bulundu")
                
                found_in_page = 0
                for link in links:
                    link_text = link.get_text().strip()
                    href = link.get('href', '')
                    
                    # Pattern kontrolü - hem "ihalelerin" hem "ihalenin" versiyonlarını yakala
                    pattern1 = r'\d{1,2}\s+\w+\s+\d{4}\s+Tarihinde\s+Gerçekleştirilen\s+İhalelerin\s+Sonuçlarına\s+İlişkin\s+Basın\s+Duyurusu'
                    pattern2 = r'\d{1,2}\s+\w+\s+\d{4}\s+Tarihinde\s+Gerçekleştirilen\s+İhalenin\s+Sonuçlarına\s+İlişkin\s+Basın\s+Duyurusu'
                    
                    if (re.search(pattern1, link_text, re.IGNORECASE) or 
                        re.search(pattern2, link_text, re.IGNORECASE)):
                        full_url = urljoin(self.base_url, href)
                        if full_url not in auction_urls:
                            auction_urls.append(full_url)
                            found_in_page += 1
                            logger.info(f"✓ İhale duyurusu bulundu: {link_text}")
                
                logger.info(f"Sayfa {page}'de {found_in_page} ihale duyurusu bulundu")
                time.sleep(2)
                
            except Exception as e:
                logger.error(f"Sayfa {page} işlenirken hata: {str(e)}")
                continue
        
        logger.info(f"Toplam {len(auction_urls)} ihale duyurusu bulundu")
        return auction_urls
    
    def get_pdf_url_from_announcement(self, announcement_url: str) -> Optional[str]:
        """
        Duyuru sayfasından PDF URL'sini çıkarır
        """
        try:
            logger.info(f"PDF linkini arıyor: {announcement_url}")
            
            if not self.driver:
                self.init_driver()
            
            self.driver.get(announcement_url)
            WebDriverWait(self.driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            time.sleep(3)
            
            # Spesifik XPath ile dene
            specific_xpath = "/html/body/div[1]/div[2]/article/div[2]/div/div/div[2]/p"
            
            try:
                paragraph_element = self.driver.find_element(By.XPATH, specific_xpath)
                link_elements = paragraph_element.find_elements(By.TAG_NAME, "a")
                
                for link in link_elements:
                    link_text = link.text.strip()
                    href = link.get_attribute('href')
                    
                    if "tıklayınız" in link_text.lower() and href:
                        logger.info(f"✓ PDF linki bulundu: {href}")
                        return href
                        
            except NoSuchElementException:
                logger.warning(f"Spesifik XPath bulunamadı")
            
            # Alternatif: Tüm PDF linklerini bul
            all_links = self.driver.find_elements(By.TAG_NAME, "a")
            for link in all_links:
                href = link.get_attribute('href')
                text = link.text.strip()
                
                if href and href.lower().endswith('.pdf') and ("ihale" in href.lower() or "duyuru" in text.lower()):
                    logger.info(f"✓ Alternatif PDF linki bulundu: {href}")
                    return href
                    
        except Exception as e:
            logger.error(f"PDF URL çıkarılırken hata: {str(e)}")
            
        return None
    
    def get_strategy_document_url(self) -> Optional[str]:
        """
        İç Borçlanma Stratejisi PDF URL'sini bulur
        """
        if not self.driver:
            self.init_driver()
            
        logger.info("İç Borçlanma Stratejisi dökumanı aranıyor...")
        
        for page in range(1, min(self.max_pages + 1, 3)):  # En fazla 3 sayfa kontrol et
            try:
                url = f"{self.category_url}{page}"
                self.driver.get(url)
                
                WebDriverWait(self.driver, 15).until(
                    EC.presence_of_element_located((By.TAG_NAME, "body"))
                )
                
                time.sleep(3)
                
                soup = BeautifulSoup(self.driver.page_source, 'html.parser')
                links = soup.find_all('a', href=True)
                
                for link in links:
                    link_text = link.get_text().strip()
                    href = link.get('href', '')
                    
                    # "İç Borçlanma Stratejisi" içeren linkleri bul
                    if "İç Borçlanma Stratejisi" in link_text:
                        full_url = urljoin(self.base_url, href)
                        logger.info(f"✓ İç Borçlanma Stratejisi bulundu: {link_text}")
                        
                        # PDF linkini al
                        self.driver.get(full_url)
                        time.sleep(3)
                        
                        # Aynı XPath ile PDF'i bul
                        try:
                            paragraph_element = self.driver.find_element(
                                By.XPATH, "/html/body/div[1]/div[2]/article/div[2]/div/div/div[2]/p"
                            )
                            link_elements = paragraph_element.find_elements(By.TAG_NAME, "a")
                            
                            for pdf_link in link_elements:
                                pdf_href = pdf_link.get_attribute('href')
                                if pdf_href and pdf_href.endswith('.pdf'):
                                    logger.info(f"✓ Strateji PDF'i bulundu: {pdf_href}")
                                    return pdf_href
                        except:
                            pass
                            
                        # Alternatif: Tüm PDF linklerini kontrol et
                        all_links_on_page = self.driver.find_elements(By.TAG_NAME, "a")
                        for pdf_link in all_links_on_page:
                            pdf_href = pdf_link.get_attribute('href')
                            if pdf_href and pdf_href.lower().endswith('.pdf') and ("strateji" in pdf_href.lower() or "tıklayınız" in pdf_link.text.lower()):
                                logger.info(f"✓ Strateji PDF'i bulundu (alternatif): {pdf_href}")
                                return pdf_href
            
            except Exception as e:
                logger.error(f"Strateji dökumanı aranırken hata: {str(e)}")
                
        logger.warning("İç Borçlanma Stratejisi dökumanı bulunamadı")
        return None
    
    def extract_data_from_pdf(self, pdf_url: str) -> List[Dict]:
        """
        PDF'den ihale verilerini çıkarır - GELİŞTİRİLMİŞ VERSİYON
        """
        try:
            logger.info(f"PDF indiriliyor: {pdf_url}")
            response = self.session.get(pdf_url, timeout=200)
            response.raise_for_status()
            
            pdf_file = io.BytesIO(response.content)
            pdf_reader = PyPDF2.PdfReader(pdf_file)
            
            all_auction_data = []
            
            # Her sayfayı ayrı ayrı işle
            for page_num, page in enumerate(pdf_reader.pages):
                page_text = page.extract_text()
                logger.info(f"Sayfa {page_num + 1} işleniyor...")
                
                # Bu sayfadaki ihaleleri çıkar
                page_auctions = self.parse_auction_data_improved(page_text)
                all_auction_data.extend(page_auctions)
            
            logger.info(f"✓ PDF'den toplam {len(all_auction_data)} ihale verisi çıkarıldı")
            return all_auction_data
            
        except Exception as e:
            logger.error(f"PDF işlenirken hata: {str(e)}")
            return []
    
    def parse_auction_data_improved(self, text: str) -> List[Dict]:
        """
        PDF metninden ihale verilerini çıkarır - GELİŞTİRİLMİŞ VERSİYON
        """
        auction_data = []
        
        try:
            # Metni bloklara ayır (her ihale bir blok)
            # ISIN kodları ile blokları tanımla
            isin_pattern = r'(TR[A-Z0-9]{10})'
            
            # Tüm ISIN kodlarını ve pozisyonlarını bul
            isin_matches = list(re.finditer(isin_pattern, text))
            
            for i, match in enumerate(isin_matches):
                current_auction = {field: '' for field in self.fields}
                
                # ISIN kodunu al
                isin = match.group(1)
                current_auction['ISIN'] = isin
                
                # Bu ihale için metin bloğunu belirle
                start_pos = match.start()
                if i < len(isin_matches) - 1:
                    end_pos = isin_matches[i + 1].start()
                else:
                    end_pos = len(text)
                
                block_text = text[start_pos:end_pos]
                
                # Blok içindeki verileri çıkar
                self.extract_auction_details(block_text, current_auction)
                
                if current_auction['ISIN']:
                    auction_data.append(current_auction)
                    logger.info(f"✓ İhale verisi çıkarıldı: {current_auction['ISIN']} - {current_auction['Senet Tanımı']}")
                    
        except Exception as e:
            logger.error(f"Veri ayrıştırırken hata: {str(e)}")
        
        return auction_data
    
    def extract_auction_details(self, block_text: str, auction_data: Dict):
        """
        Bir ihale bloğundan detayları çıkar
        """
        lines = block_text.split('\n')
        
        # Patterns for extraction
        patterns = {
            'Senet Tanımı': r'Senet Tanımı\s*:\s*(.+?)(?:Ortalama|$)',
            'İhale Tarihi': r'İhale Tarihi\s*:\s*(\d{2}\.\d{2}\.\d{4})',
            'Valör Tarihi': r'(?:İhraç|Valör)\s*\(?Valör\)?\s*Tarihi\s*:\s*(\d{2}\.\d{2}\.\d{4})', # İhraç (Valör) veya Valör Tarihi
            'İtfa Tarihi': r'(?:Vade|İtfa)\s*Tarihi\s*:\s*(\d{2}\.\d{2}\.\d{4})' # Vade Tarihi veya İtfa Tarihi
        }
        
        # Basit alanları çıkar
        for field, pattern in patterns.items():
            match = re.search(pattern, block_text, re.IGNORECASE | re.DOTALL)
            if match:
                value = match.group(1).strip().replace('\n', ' ')
                # Senet tanımından gereksiz kısımları temizle
                if field == 'Senet Tanımı':
                    value = re.sub(r'(Ortalama|En Düşük|En Yüksek).*', '', value).strip()
                auction_data[field] = value
        
        # Vade hesapla (yıl cinsinden)
        if auction_data.get('Valör Tarihi') and auction_data.get('İtfa Tarihi'):
            try:
                valor_date = pd.to_datetime(auction_data['Valör Tarihi'], format='%d.%m.%Y')
                maturity_date = pd.to_datetime(auction_data['İtfa Tarihi'], format='%d.%m.%Y')
                days_diff = (maturity_date - valor_date).days
                years = round(days_diff / 365.25, 2)
                auction_data['Vade (Yıl)'] = years
            except:
                auction_data['Vade (Yıl)'] = ''
        
        # Sayısal değerleri çıkar
        self.extract_numeric_values(block_text, auction_data)
    
    def extract_numeric_values(self, block_text: str, auction_data: Dict):
        """
        Blok metninden sayısal değerleri çıkar - GELİŞTİRİLMİŞ VERSİYON
        """
        # Miktar bölümünü bul ve çıkar
        miktar_section = re.search(r'Miktar \(Net, Milyon TL\)(.*?)(?:Faiz Oranları|Fiyatlar|İhraç Sonrası|$)', 
                                   block_text, re.DOTALL | re.IGNORECASE)
        
        if miktar_section:
            miktar_text = miktar_section.group(1)
            
            # ROT satırını bul - Toplam
            rot_match = re.search(r'ROT\s*:\s*([\d.,-]+)\s+([\d.,-]+)', miktar_text)
            if rot_match:
                auction_data['ROT Toplam(Teklif)'] = rot_match.group(1).replace('.', '').replace(',', '.')
                auction_data['ROT Toplam(Gerçekleşme)'] = rot_match.group(2).replace('.', '').replace(',', '.')
            
            # ROT - Kamu Kurumları
            rot_kamu_match = re.search(r'Kamu Kurumları\s*:\s*([\d.,-]+)\s+([\d.,-]+)', miktar_text)
            if rot_kamu_match:
                auction_data['ROT Kamu(Teklif)'] = rot_kamu_match.group(1).replace('.', '').replace(',', '.')
                auction_data['ROT Kamu(Gerçekleşme)'] = rot_kamu_match.group(2).replace('.', '').replace(',', '.')
            
            # ROT - Piyasa Yapıcılar
            rot_piyasa_pattern = r'(?:ROT.*?)?Piyasa Yapıcılar\s*:\s*([\d.,-]+)\s+([\d.,-]+)'
            rot_piyasa_matches = re.findall(rot_piyasa_pattern, miktar_text)
            if rot_piyasa_matches:
                # İlk eşleşme ROT için
                auction_data['ROT Piyasa Yapıcılar(Teklif)'] = rot_piyasa_matches[0][0].replace('.', '').replace(',', '.')
                auction_data['ROT Piyasa Yapıcılar(Gerçekleşme)'] = rot_piyasa_matches[0][1].replace('.', '').replace(',', '.')
                
                # ROT Piyasa Yapıcılar Kabul Oranı hesapla
                try:
                    teklif = float(auction_data['ROT Piyasa Yapıcılar(Teklif)'])
                    gerceklesme = float(auction_data['ROT Piyasa Yapıcılar(Gerçekleşme)'])
                    if teklif > 0:
                        kabul_orani = round((gerceklesme / teklif) * 100, 1)
                        auction_data['ROT Piyasa Yapıcılar Kabul Oranı (%)'] = kabul_orani
                except:
                    pass
            
            # İhale satırını bul - Toplam olarak al
            ihale_match = re.search(r'İhale\s*:\s*([\d.,-]+)\s+([\d.,-]+)', miktar_text)
            if ihale_match:
                auction_data['İhale(Teklif)'] = ihale_match.group(1).replace('.', '').replace(',', '.')
                auction_data['İhale(Gerçekleşme)'] = ihale_match.group(2).replace('.', '').replace(',', '.')
                
                # İhale Kabul Oranı hesapla
                try:
                    teklif = float(auction_data['İhale(Teklif)'])
                    gerceklesme = float(auction_data['İhale(Gerçekleşme)'])
                    if teklif > 0:
                        kabul_orani = round((gerceklesme / teklif) * 100, 1)
                        auction_data['İhale Kabul Oranı (%)'] = kabul_orani
                except:
                    pass
            
            # Toplam satırını bul
            toplam_match = re.search(r'Toplam\s*:\s*([\d.,-]+)\s+([\d.,-]+)', miktar_text)
            if toplam_match:
                auction_data['Toplam(Teklif)'] = toplam_match.group(1).replace('.', '').replace(',', '.')
                auction_data['Toplam(Gerçekleşme)'] = toplam_match.group(2).replace('.', '').replace(',', '.')
        
        # Faiz değerlerini bul
        faiz_patterns = {
            'Ortalama Yıllık Basit': r'Ortalama Yıllık Basit\s*:\s*([\d.,-]+)\s+([\d.,-]+)',
            'Ortalama Yıllık Bileşik': r'Ortalama Yıllık Bileşik\s*:\s*([\d.,-]+)\s+([\d.,-]+)',
            'En Düşük Yıllık Bileşik': r'En Düşük Yıllık Bileşik\s*:\s*([\d.,-]+)\s+([\d.,-]+)',
            'En Yüksek Yıllık Bileşik': r'En Yüksek Yıllık Bileşik\s*:\s*([\d.,-]+)\s+([\d.,-]+)'
        }
        
        for field_base, pattern in faiz_patterns.items():
            match = re.search(pattern, block_text, re.IGNORECASE)
            if match:
                auction_data[f'{field_base}(Teklif)'] = match.group(1).replace(',', '.')
                auction_data[f'{field_base}(Gerçekleşme)'] = match.group(2).replace(',', '.')
        
        # Fiyat değerlerini bul
        fiyat_patterns = {
            'Ortalama Fiyat': r'Ortalama Fiyat\s*:\s*([\d.,-]+)\s+([\d.,-]+)',
            'En Yüksek Fiyat': r'En Yüksek Fiyat\s*:\s*([\d.,-]+)\s+([\d.,-]+)',
            'En Düşük Fiyat': r'En Düşük Fiyat\s*:\s*([\d.,-]+)\s+([\d.,-]+)'
        }
        
        for field_base, pattern in fiyat_patterns.items():
            match = re.search(pattern, block_text, re.IGNORECASE)
            if match:
                auction_data[f'{field_base}(Teklif)'] = match.group(1).replace('.', '').replace(',', '.')
                auction_data[f'{field_base}(Gerçekleşme)'] = match.group(2).replace('.', '').replace(',', '.')
    
    def extract_strategy_data(self, pdf_url: str) -> Dict[str, float]:
        """
        İç Borçlanma Stratejisi PDF'inden hedef borçlanma miktarlarını çıkarır
        
        Returns:
            Dict: Ay bazında hedef borçlanma miktarları (milyar TL)
        """
        try:
            logger.info(f"Strateji PDF'i indiriliyor: {pdf_url}")
            response = self.session.get(pdf_url, timeout=20)
            response.raise_for_status()
            
            pdf_file = io.BytesIO(response.content)
            pdf_reader = PyPDF2.PdfReader(pdf_file)
            
            strategy_data = {}
            
            # PDF'deki tüm sayfaları oku
            full_text = ""
            for page in pdf_reader.pages:
                full_text += page.extract_text() + "\n"
            
            # Başlıktan dönem bilgisini al
            title_pattern = r'(\w+)\s*[-–]\s*(\w+)\s+\d{4}'
            title_match = re.search(title_pattern, full_text, re.IGNORECASE)
            
            months = ['Ocak', 'Şubat', 'Mart', 'Nisan', 'Mayıs', 'Haziran', 
                      'Temmuz', 'Ağustos', 'Eylül', 'Ekim', 'Kasım', 'Aralık']
            
            month_list = []
            if title_match:
                ### DÜZELTME 1: Ay isimlerini standart formata getirmek için .strip().title() eklendi ###
                start_month = title_match.group(1).strip().title()
                end_month = title_match.group(2).strip().title()
                logger.info(f"Dönem: {start_month} - {end_month}")
                
                # Ayları belirle
                try:
                    start_idx = months.index(start_month)
                    end_idx = months.index(end_month)
                    
                    if end_idx >= start_idx:
                        month_list = months[start_idx:end_idx+1]
                    else: # Yıl döndüğü durumlar için (örn: Aralık-Şubat)
                        month_list = months[start_idx:] + months[:end_idx+1]
                    logger.info(f"Ay listesi: {month_list}")
                except ValueError:
                    logger.error(f"Ay listesinde bulunamadı: {start_month} veya {end_month}")
            
            # Sadece tablodan veri al - "Piyasadan İhale Yoluyla İç Borçlanma" satırı
            # PDF formatı değişebileceğinden daha esnek bir pattern
            pattern = r'Piyasadan(?:\s+İhale)?\s+Yoluyla\s+İç\s+Borçlanma\s+([\d.,]+)\s+([\d.,]+)\s+([\d.,]+)'
            match = re.search(pattern, full_text)
                
            if match and month_list:
                logger.info("Piyasadan İhale Yoluyla İç Borçlanma satırı bulundu")
                
                # Her ay için değerleri al
                values = [match.group(1), match.group(2), match.group(3)]
                
                for i, month in enumerate(month_list[:3]):
                    if i < len(values):
                        # Türkçe sayı formatını düzelt (nokta binlik ayracı, virgül ondalık)
                        value_str = values[i].replace('.', '').replace(',', '.')
                        try:
                            strategy_data[month] = float(value_str)
                            logger.info(f"{month} hedef borçlanma: {strategy_data[month]} milyar TL")
                        except ValueError:
                            logger.error(f"Sayıya dönüştürülemedi: {values[i]}")
            elif not match:
                logger.warning("Piyasadan İhale Yoluyla İç Borçlanma satırı bulunamadı")
            
            logger.info(f"Toplam {len(strategy_data)} ay için hedef bulundu")
            return strategy_data
            
        except Exception as e:
            logger.error(f"Strateji PDF'si işlenirken hata: {str(e)}")
            return {}

    def analyze_borrowing_performance(self, auction_df: pd.DataFrame, strategy_data: Dict[str, float]) -> pd.DataFrame:
        """
        Hedef ve gerçekleşen borçlanmaları karşılaştırır
        """
        if strategy_data and not auction_df.empty:
            # Ay bazında gerçekleşen borçlanmaları topla
            monthly_realized = {}
            
            # Tarih formatını kontrol et
            if 'İhale Tarihi' not in auction_df.columns:
                logger.error("DataFrame'de 'İhale Tarihi' sütunu bulunamadı.")
                return pd.DataFrame()
            
            # 'İhale Tarihi' sütunundaki null değerleri atla
            df_filtered = auction_df.dropna(subset=['İhale Tarihi', 'Toplam(Gerçekleşme)'])

            for _, row in df_filtered.iterrows():
                try:
                    date_obj = pd.to_datetime(row['İhale Tarihi'], format='%d.%m.%Y')
                    
                    month_names = {
                        1: 'Ocak', 2: 'Şubat', 3: 'Mart', 4: 'Nisan',
                        5: 'Mayıs', 6: 'Haziran', 7: 'Temmuz', 8: 'Ağustos',
                        9: 'Eylül', 10: 'Ekim', 11: 'Kasım', 12: 'Aralık'
                    }
                    month = month_names.get(date_obj.month)
                    
                    total_realized = pd.to_numeric(row['Toplam(Gerçekleşme)'], errors='coerce')
                    
                    if month and not pd.isna(total_realized):
                        monthly_realized.setdefault(month, 0)
                        monthly_realized[month] += total_realized
                        
                except Exception as e:
                    logger.debug(f"Tarih veya sayı parse hatası: {row.get('İhale Tarihi')} - {str(e)}")
            
            # Karşılaştırma tablosu oluştur
            comparison_data = []
            
            for month, target in strategy_data.items():
                realized = monthly_realized.get(month, 0) / 1000  # Milyon TL'den milyar TL'ye çevir
                
                comparison_data.append({
                    'Ay': month,
                    'Hedef Borçlanma (Milyar TL)': target,
                    'Gerçekleşen Borçlanma (Milyar TL)': round(realized, 2),
                    'Fark (Milyar TL)': round(realized - target, 2),
                    'Gerçekleşme Oranı (%)': round((realized / target * 100) if target > 0 else 0, 1)
                })
            
            if not comparison_data:
                logger.warning("Karşılaştırma verisi oluşturulamadı. Aylık gerçekleşme bulunamadı mı?")
                return pd.DataFrame()

            comparison_df = pd.DataFrame(comparison_data)
            
            # Özet istatistikler
            logger.info("\n" + "="*60)
            logger.info("BORÇLANMA HEDEFLERİ ANALİZİ")
            logger.info("="*60)
            
            for _, row in comparison_df.iterrows():
                status = "✓" if row['Fark (Milyar TL)'] >= 0 else "✗"
                logger.info(f"{status} {row['Ay']}: Hedef {row['Hedef Borçlanma (Milyar TL)']} Milyar TL, "
                            f"Gerçekleşen {row['Gerçekleşen Borçlanma (Milyar TL)']} Milyar TL "
                            f"(Fark: {row['Fark (Milyar TL)']} Milyar TL, %{row['Gerçekleşme Oranı (%)']})")
            
            # Son ay için kalan borçlanma
            last_month_target = list(strategy_data.keys())[-1]
            if last_month_target in comparison_df['Ay'].values:
                last_row = comparison_df[comparison_df['Ay'] == last_month_target].iloc[0]
                remaining = last_row['Hedef Borçlanma (Milyar TL)'] - last_row['Gerçekleşen Borçlanma (Milyar TL)']
                if remaining > 0:
                    logger.info(f"\n⚠ {last_month_target} ayı için kalan borçlanma hedefi: {round(remaining, 2)} Milyar TL")
            
            return comparison_df
        
        return pd.DataFrame()

    def scrape_all_auctions(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Tüm ihale verilerini çeker ve analiz eder
        """
        all_auction_data = []
        comparison_df = pd.DataFrame()
        
        try:
            logger.info("İhale verisi çekme işlemi başlatılıyor...")
            
            # İç Borçlanma Stratejisi'ni bul ve analiz et
            strategy_data = {}
            if self.analyze_strategy:
                strategy_url = self.get_strategy_document_url()
                if strategy_url:
                    strategy_data = self.extract_strategy_data(strategy_url)
                    if not strategy_data:
                        logger.warning("Strateji PDF'inden veri çıkarılamadı")
                else:
                    logger.warning("İç Borçlanma Stratejisi dökumanı bulunamadı, hedef analizi yapılamayacak")
            
            # Duyuru URL'lerini al
            announcement_urls = self.get_auction_announcement_urls()
            
            if not announcement_urls:
                logger.error("Hiç duyuru URL'si bulunamadı!")
                return pd.DataFrame(columns=self.fields), pd.DataFrame()
            
            logger.info(f"Toplam {len(announcement_urls)} duyuru bulundu, işleme başlanıyor...")
            
            for i, announcement_url in enumerate(announcement_urls, 1):
                logger.info(f"\n{'='*60}")
                logger.info(f"İŞLENİYOR ({i}/{len(announcement_urls)}): {announcement_url}")
                logger.info(f"{'='*60}")
                
                # PDF URL'sini al
                pdf_url = self.get_pdf_url_from_announcement(announcement_url)
                
                if pdf_url:
                    logger.info(f"✓ PDF bulundu: {pdf_url}")
                    
                    # PDF'den veri çıkar
                    auction_data = self.extract_data_from_pdf(pdf_url)
                    
                    if auction_data:
                        all_auction_data.extend(auction_data)
                        logger.info(f"✓ {len(auction_data)} ihale verisi eklendi")
                    else:
                        logger.warning("⚠ Bu PDF'den veri çıkarılamadı")
                    
                    time.sleep(3)  # Rate limiting
                else:
                    logger.error(f"✗ PDF URL bulunamadı: {announcement_url}")
            
        except Exception as e:
            logger.error(f"Ana işlem sırasında hata: {str(e)}")
        finally:
            self.close_driver()
        
        # DataFrame oluştur
        if all_auction_data:
            df = pd.DataFrame(all_auction_data)
            
            # Sayısal kolonları düzelt
            numeric_columns = [col for col in df.columns if any(x in col for x in ['Teklif', 'Gerçekleşme', 'Fiyat', 'Oranı', 'Vade'])]
            for col in numeric_columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            
            logger.info(f"\n{'='*60}")
            logger.info(f"✓ BAŞARIYLA TAMAMLANDI!")
            logger.info(f"✓ Toplam {len(df)} ihale verisi çekildi")
            logger.info(f"{'='*60}")
            
            # Hedef vs Gerçekleşme analizi
            if strategy_data and self.analyze_strategy:
                logger.info(f"Strateji verisi mevcut: {len(strategy_data)} ay")
                comparison_df = self.analyze_borrowing_performance(df, strategy_data)
                if comparison_df.empty:
                    logger.warning("Karşılaştırma tablosu boş döndü!")
            else:
                logger.warning(f"Hedef analizi yapılamadı. strategy_data: {len(strategy_data) if strategy_data else 0}, analyze_strategy: {self.analyze_strategy}")
            
            return df, comparison_df
        else:
            logger.error("✗ HİÇ VERİ ÇEKİLEMEDİ!")
            return pd.DataFrame(columns=self.fields), pd.DataFrame()

    def calculate_weighted_average_maturity(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Aylık ağırlıklı ortalama vade hesaplar
        """
        if df.empty or 'İhale Tarihi' not in df.columns:
            return pd.DataFrame()
        
        df_copy = df.copy()
        # İhale tarihini datetime'a çevir
        df_copy['İhale Tarihi'] = pd.to_datetime(df_copy['İhale Tarihi'], format='%d.%m.%Y', errors='coerce')
        df_copy.dropna(subset=['İhale Tarihi', 'Vade (Yıl)', 'Toplam(Gerçekleşme)'], inplace=True)

        if df_copy.empty:
            return pd.DataFrame()

        # Ay ve yıl bilgisi ekle
        df_copy['Yıl-Ay'] = df_copy['İhale Tarihi'].dt.to_period('M')
        
        # Ağırlıklı vade hesapla: (Vade * Gerçekleşen Miktar) / Toplam Gerçekleşen Miktar
        monthly_wam = []
        
        for period in df_copy['Yıl-Ay'].unique():
            if pd.notna(period):
                month_data = df_copy[df_copy['Yıl-Ay'] == period]
                
                total_realized = month_data['Toplam(Gerçekleşme)'].sum()
                
                if total_realized > 0:
                    weighted_sum = (month_data['Vade (Yıl)'] * month_data['Toplam(Gerçekleşme)']).sum()
                    wam = weighted_sum / total_realized
                    
                    monthly_wam.append({
                        'Dönem': period,
                        'Tarih': period.to_timestamp(),
                        'Ağırlıklı Ortalama Vade (Yıl)': round(wam, 2),
                        'Toplam İhraç (Milyon TL)': round(total_realized, 2),
                        'İhale Sayısı': len(month_data)
                    })
        
        if not monthly_wam:
            return pd.DataFrame()

        wam_df = pd.DataFrame(monthly_wam)
        
        # Tarihe göre sırala
        wam_df = wam_df.sort_values('Tarih')
            
        # 3 aylık hareketli ortalama hesapla
        wam_df['3 Aylık Ortalama Vade'] = wam_df['Ağırlıklı Ortalama Vade (Yıl)'].rolling(window=3, min_periods=1).mean().round(2)
        
        return wam_df

    def create_maturity_charts(self, wam_df: pd.DataFrame, output_file: str = "vade_analizi.html"):
        """
        Ağırlıklı ortalama vade grafiklerini oluşturur
        """
        if wam_df.empty:
            logger.warning("Vade verisi bulunamadı, grafik oluşturulamadı")
            return
        
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=('Aylık Ağırlıklı Ortalama Vade ve İhraç Miktarı', '3 Aylık Hareketli Ortalama Vade Trendi'),
            vertical_spacing=0.15,
            specs=[[{"secondary_y": True}], [{"secondary_y": False}]]
        )
        
        # Aylık vade grafiği
        fig.add_trace(
            go.Scatter(
                x=wam_df['Tarih'], y=wam_df['Ağırlıklı Ortalama Vade (Yıl)'],
                mode='lines+markers', name='Aylık Vade (Yıl)',
                line=dict(color='#1f77b4', width=2), marker=dict(size=8),
                hovertemplate='<b>%{x|%B %Y}</b><br>Ağırlıklı Vade: %{y:.2f} yıl<extra></extra>'
            ),
            row=1, col=1, secondary_y=False
        )
        
        # İhraç miktarı (ikincil y ekseni)
        fig.add_trace(
            go.Bar(
                x=wam_df['Tarih'], y=wam_df['Toplam İhraç (Milyon TL)'],
                name='İhraç Miktarı (Milyon TL)', yaxis='y2',
                marker_color='lightgray', opacity=0.5,
                hovertemplate='<b>%{x|%B %Y}</b><br>İhraç: %{y:,.0f} Milyon TL<extra></extra>'
            ),
            row=1, col=1, secondary_y=True
        )
        
        # 3 aylık ortalama grafiği
        fig.add_trace(
            go.Scatter(
                x=wam_df['Tarih'], y=wam_df['3 Aylık Ortalama Vade'],
                mode='lines+markers', name='3 Aylık Ortalama',
                line=dict(color='#ff7f0e', width=3, dash='dash'), marker=dict(size=6),
                hovertemplate='<b>%{x|%B %Y}</b><br>3 Aylık Ort: %{y:.2f} yıl<extra></extra>'
            ),
            row=2, col=1
        )
        
        # Layout ayarları
        fig.update_layout(
            title={'text': 'Türkiye Hazinesi - Borçlanma Vade Analizi', 'x': 0.5, 'xanchor': 'center', 'font': {'size': 20}},
            height=800, showlegend=True, hovermode='x unified', template='plotly_white',
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        
        # Y ekseni ayarları
        fig.update_yaxes(title_text="Vade (Yıl)", row=1, col=1, secondary_y=False)
        fig.update_yaxes(title_text="İhraç Miktarı (Milyon TL)", row=1, col=1, secondary_y=True, showgrid=False)
        fig.update_yaxes(title_text="Vade (Yıl)", row=2, col=1)
        
        # X ekseni ayarları
        fig.update_xaxes(title_text="", row=1, col=1)
        fig.update_xaxes(title_text="Tarih", row=2, col=1)

        fig.write_html(output_file)
        logger.info(f"✓ Vade analizi grafikleri {output_file} dosyasına kaydedildi")
        
        # Özet istatistikler
        logger.info("\n" + "="*60)
        logger.info("VADE ANALİZİ ÖZETİ")
        logger.info("="*60)
        logger.info(f"Ortalama Vade: {wam_df['Ağırlıklı Ortalama Vade (Yıl)'].mean():.2f} yıl")
        logger.info(f"En Kısa Vade: {wam_df['Ağırlıklı Ortalama Vade (Yıl)'].min():.2f} yıl")
        logger.info(f"En Uzun Vade: {wam_df['Ağırlıklı Ortalama Vade (Yıl)'].max():.2f} yıl")
        if not wam_df.empty:
            logger.info(f"Son 3 Ay Ortalaması: {wam_df['3 Aylık Ortalama Vade'].iloc[-1]:.2f} yıl")
        
        return fig

    ### DÜZELTME 2: Kod bloğu, class içinde bir metod haline getirildi ###
    def save_to_excel(self, df: pd.DataFrame, comparison_df: Optional[pd.DataFrame], wam_df: Optional[pd.DataFrame], filename: str = "hazine_ihale_verileri.xlsx"):
        """
        DataFrame'leri Excel dosyasına kaydet
        """
        try:
            with pd.ExcelWriter(filename, engine='openpyxl') as writer:
                # İhale verileri sayfası
                if df is not None and not df.empty:
                    df.to_excel(writer, index=False, sheet_name='İhale Verileri')
                    worksheet = writer.sheets['İhale Verileri']
                    for idx, col in enumerate(df.columns):
                        max_len = max(df[col].astype(str).map(len).max(), len(str(col))) + 2
                        worksheet.column_dimensions[worksheet.cell(1, idx + 1).column_letter].width = min(max_len, 50)

                # Eğer karşılaştırma verisi varsa ekle
                if comparison_df is not None and not comparison_df.empty:
                    comparison_df.to_excel(writer, index=False, sheet_name='Hedef vs Gerçekleşme')
                    worksheet = writer.sheets['Hedef vs Gerçekleşme']
                    for idx, col in enumerate(comparison_df.columns):
                        max_len = max(comparison_df[col].astype(str).map(len).max(), len(str(col))) + 2
                        worksheet.column_dimensions[worksheet.cell(1, idx + 1).column_letter].width = min(max_len, 40)
                
                # Eğer vade analizi varsa ekle
                if wam_df is not None and not wam_df.empty:
                    wam_df.to_excel(writer, index=False, sheet_name='Vade Analizi')
                    worksheet = writer.sheets['Vade Analizi']
                    for idx, col in enumerate(wam_df.columns):
                        max_len = max(wam_df[col].astype(str).map(len).max(), len(str(col))) + 2
                        worksheet.column_dimensions[worksheet.cell(1, idx + 1).column_letter].width = min(max_len, 40)
            logger.info(f"✓ Veriler başarıyla {filename} dosyasına kaydedildi.")
        except Exception as e:
            logger.error(f"Excel dosyası kaydedilirken hata: {str(e)}")


def main():
    """Ana fonksiyon"""
    try:
        # Scraper'ı başlat (strateji analizi aktif, 10 sayfa)
        scraper = TreasuryAuctionScraper(max_pages=250, headless=True, analyze_strategy=True)
        
        # Tüm ihale verilerini çek ve analiz et
        df, comparison_df = scraper.scrape_all_auctions()
        
        # Sonuçları göster
        if not df.empty:
            print(f"\n{'='*60}")
            print(f"✓ BAŞARILI! Toplam {len(df)} ihale verisi çekildi")
            print(f"{'='*60}")
            
            # Özet bilgileri göster
            print("\nÇekilen ISIN kodları:")
            unique_isins = df.groupby('ISIN').agg({
                'Senet Tanımı': 'first',
                'İhale Tarihi': 'count'
            }).reset_index()
            unique_isins.columns = ['ISIN', 'Senet Tanımı', 'İhale Sayısı']
            
            for idx, row in unique_isins.iterrows():
                print(f"{idx + 1}. {row['ISIN']} - {row['Senet Tanımı']} ({row['İhale Sayısı']} ihale)")
            
            # Hedef vs Gerçekleşme tablosunu göster
            if not comparison_df.empty:
                print(f"\n{'='*60}")
                print("HEDEF VS GERÇEKLEŞME ANALİZİ")
                print(f"{'='*60}")
                print(comparison_df.to_string(index=False))
                
                # Son ay için kalan borçlanma
                last_row = comparison_df.iloc[-1]
                remaining = last_row['Hedef Borçlanma (Milyar TL)'] - last_row['Gerçekleşen Borçlanma (Milyar TL)']
                if remaining > 0:
                    print(f"\n⚠ {last_row['Ay']} ayı için kalan borçlanma hedefi: {round(remaining, 2)} Milyar TL")
            else:
                print("\n⚠ Hedef karşılaştırma tablosu oluşturulamadı!")
            
            # Ağırlıklı ortalama vade analizi
            print(f"\n{'='*60}")
            print("AĞIRLIKLI ORTALAMA VADE ANALİZİ")
            print(f"{'='*60}")
            
            wam_df = scraper.calculate_weighted_average_maturity(df)
            
            if not wam_df.empty:
                print(wam_df[['Dönem', 'Ağırlıklı Ortalama Vade (Yıl)', '3 Aylık Ortalama Vade', 'Toplam İhraç (Milyon TL)']].to_string(index=False))
                
                # Vade grafiklerini oluştur
                scraper.create_maturity_charts(wam_df)
                print("\n✓ Vade analizi grafikleri 'vade_analizi.html' dosyasına kaydedildi")
            else:
                print("Vade analizi için yeterli veri bulunamadı")
            
            # Excel'e kaydet (3 sayfa: İhale Verileri + Hedef vs Gerçekleşme + Vade Analizi)
            scraper.save_to_excel(df, comparison_df, wam_df)
            
            # CSV'ye de kaydet
            df.to_csv("hazine_ihale_verileri.csv", index=False, encoding='utf-8-sig')
            if not comparison_df.empty:
                comparison_df.to_csv("hazine_hedef_gerceklesme.csv", index=False, encoding='utf-8-sig')
            if not wam_df.empty:
                wam_df.to_csv("hazine_vade_analizi.csv", index=False, encoding='utf-8-sig')
            
            print(f"\n✓ Veriler kaydedildi:")
            print(f"  - hazine_ihale_verileri.xlsx")
            print(f"  - hazine_ihale_verileri.csv")
            if not comparison_df.empty:
                print(f"  - hazine_hedef_gerceklesme.csv")
            if not wam_df.empty:
                print(f"  - hazine_vade_analizi.csv")
                print(f"  - vade_analizi.html (interaktif grafikler)")
        else:
            print(f"\n{'='*60}")
            print(f"✗ HİÇ VERİ ÇEKİLEMEDİ!")
            print(f"{'='*60}")
            
    except Exception as e:
        print(f"\n{'='*60}")
        print(f"✗ HATA: {str(e)}")
        print(f"{'='*60}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

2025-07-23 22:33:06,140 - INFO - İhale verisi çekme işlemi başlatılıyor...
2025-07-23 22:33:06,818 - WARNING - The chromedriver version (137.0.7151.119) detected in PATH at /usr/local/bin/chromedriver might not be compatible with the detected chrome version (138.0.7204.159); currently, chromedriver 138.0.7204.168 is recommended for chrome 138.*, so it is advised to delete the driver in PATH and retry
2025-07-23 22:33:08,071 - INFO - Chrome driver başlatıldı
2025-07-23 22:33:08,071 - INFO - İç Borçlanma Stratejisi dökumanı aranıyor...
2025-07-23 22:33:15,123 - WARNING - İç Borçlanma Stratejisi dökumanı bulunamadı
2025-07-23 22:33:15,124 - WARNING - İç Borçlanma Stratejisi dökumanı bulunamadı, hedef analizi yapılamayacak
2025-07-23 22:33:15,124 - INFO - Sayfa 1 kontrol ediliyor...
2025-07-23 22:33:18,169 - INFO - Sayfa 1'de 122 link bulundu
2025-07-23 22:33:18,170 - INFO - Sayfa 1'de 0 ihale duyurusu bulundu
2025-07-23 22:33:20,176 - INFO - Sayfa 2 kontrol ediliyor...
2025-07-23 22:33:23

2025-07-23 22:34:55,419 - INFO - Sayfa 20'de 2 ihale duyurusu bulundu
2025-07-23 22:34:57,424 - INFO - Sayfa 21 kontrol ediliyor...
2025-07-23 22:35:00,513 - INFO - Sayfa 21'de 127 link bulundu
2025-07-23 22:35:00,514 - INFO - Sayfa 21'de 0 ihale duyurusu bulundu
2025-07-23 22:35:02,518 - INFO - Sayfa 22 kontrol ediliyor...
2025-07-23 22:35:05,608 - INFO - Sayfa 22'de 127 link bulundu
2025-07-23 22:35:05,608 - INFO - ✓ İhale duyurusu bulundu: 12 Kasım 2024 Tarihinde Gerçekleştirilen İhalenin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:35:05,609 - INFO - Sayfa 22'de 1 ihale duyurusu bulundu
2025-07-23 22:35:07,610 - INFO - Sayfa 23 kontrol ediliyor...
2025-07-23 22:35:10,702 - INFO - Sayfa 23'de 127 link bulundu
2025-07-23 22:35:10,703 - INFO - ✓ İhale duyurusu bulundu: 11 Kasım 2024 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:35:10,704 - INFO - ✓ İhale duyurusu bulundu: 5 Kasım 2024 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişki

2025-07-23 22:36:47,625 - INFO - ✓ İhale duyurusu bulundu: 22 Nisan 2024 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:36:47,625 - INFO - ✓ İhale duyurusu bulundu: 16 Nisan 2024 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:36:47,626 - INFO - Sayfa 42'de 2 ihale duyurusu bulundu
2025-07-23 22:36:49,627 - INFO - Sayfa 43 kontrol ediliyor...
2025-07-23 22:36:52,717 - INFO - Sayfa 43'de 127 link bulundu
2025-07-23 22:36:52,719 - INFO - ✓ İhale duyurusu bulundu: 15 Nisan 2024 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:36:52,719 - INFO - Sayfa 43'de 1 ihale duyurusu bulundu
2025-07-23 22:36:54,722 - INFO - Sayfa 44 kontrol ediliyor...
2025-07-23 22:36:57,816 - INFO - Sayfa 44'de 127 link bulundu
2025-07-23 22:36:57,817 - INFO - Sayfa 44'de 0 ihale duyurusu bulundu
2025-07-23 22:36:59,820 - INFO - Sayfa 45 kontrol ediliyor...
2025-07-23 22:37:02,906 - INFO - Sayfa 45'

2025-07-23 22:38:34,647 - INFO - ✓ İhale duyurusu bulundu: 12 Eylül 2023 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:38:34,647 - INFO - Sayfa 63'de 2 ihale duyurusu bulundu
2025-07-23 22:38:36,653 - INFO - Sayfa 64 kontrol ediliyor...
2025-07-23 22:38:39,730 - INFO - Sayfa 64'de 127 link bulundu
2025-07-23 22:38:39,731 - INFO - ✓ İhale duyurusu bulundu: 11 Eylül 2023 Tarihinde Gerçekleştirilen İhalenin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:38:39,732 - INFO - Sayfa 64'de 1 ihale duyurusu bulundu
2025-07-23 22:38:41,736 - INFO - Sayfa 65 kontrol ediliyor...
2025-07-23 22:38:44,829 - INFO - Sayfa 65'de 127 link bulundu
2025-07-23 22:38:44,830 - INFO - ✓ İhale duyurusu bulundu: 15 Ağustos 2023 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:38:44,831 - INFO - ✓ İhale duyurusu bulundu: 14 Ağustos 2023 Tarihinde Gerçekleştirilen İhalenin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:38:44,832 - IN

2025-07-23 22:40:13,560 - INFO - Sayfa 83 kontrol ediliyor...
2025-07-23 22:40:16,647 - INFO - Sayfa 83'de 127 link bulundu
2025-07-23 22:40:16,648 - INFO - ✓ İhale duyurusu bulundu: 17 Ocak 2023 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:40:16,648 - INFO - ✓ İhale duyurusu bulundu: 16 Ocak 2023 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:40:16,648 - INFO - Sayfa 83'de 2 ihale duyurusu bulundu
2025-07-23 22:40:18,653 - INFO - Sayfa 84 kontrol ediliyor...
2025-07-23 22:40:21,754 - INFO - Sayfa 84'de 127 link bulundu
2025-07-23 22:40:21,756 - INFO - Sayfa 84'de 0 ihale duyurusu bulundu
2025-07-23 22:40:23,757 - INFO - Sayfa 85 kontrol ediliyor...
2025-07-23 22:40:26,847 - INFO - Sayfa 85'de 127 link bulundu
2025-07-23 22:40:26,848 - INFO - ✓ İhale duyurusu bulundu: 13 Aralık 2022 Tarihinde Gerçekleştirilen İhalelerin Sonuçlarına İlişkin Basın Duyurusu
2025-07-23 22:40:26,848 - INFO - Sayfa 85'de 1 ihal

...

2025-07-23 23:34:55,980 - INFO - PDF linkini arıyor: https://www.hmb.gov.tr/duyuru/23-mayis-2017-tarihinde-gerceklestirilen-ihalelerin-sonuclarina-iliskin-basin-duyurusu
2025-07-23 23:34:59,063 - INFO - ✓ PDF linki bulundu: https://ms.hmb.gov.tr/uploads/2020/05/76_KAF2017051876ihaleduyurukuponsuz-1-1.docx
2025-07-23 23:34:59,064 - INFO - ✓ PDF bulundu: https://ms.hmb.gov.tr/uploads/2020/05/76_KAF2017051876ihaleduyurukuponsuz-1-1.docx
2025-07-23 23:34:59,064 - INFO - PDF indiriliyor: https://ms.hmb.gov.tr/uploads/2020/05/76_KAF2017051876ihaleduyurukuponsuz-1-1.docx
2025-07-23 23:34:59,239 - ERROR - PDF işlenirken hata: EOF marker not found
2025-07-23 23:34:59,240 - WARNING - ⚠ Bu PDF'den veri çıkarılamadı
2025-07-23 23:35:02,250 - INFO - 
2025-07-23 23:35:02,252 - INFO - İŞLENİYOR (304/315): https://www.hmb.gov.tr/duyuru/16-mayis-2017-tarihinde-gerceklestirilen-ihalelerin-sonuclarina-iliskin-basin-duyurusu
2025-07-23 23:35:02,253 - INFO - ================================================

2025-07-23 23:35:45,621 - INFO - ============================================================
2025-07-23 23:35:45,622 - INFO - PDF linkini arıyor: https://www.hmb.gov.tr/duyuru/14-subat-2017-tarihinde-gerceklestirilen-ihalenin-sonuclarina-iliskin-basin-duyurusu
2025-07-23 23:35:48,712 - INFO - ✓ PDF linki bulundu: https://ms.hmb.gov.tr/uploads/2020/05/27_KAF2017021427ihalesonucu10s-1.docx
2025-07-23 23:35:48,712 - INFO - ✓ PDF bulundu: https://ms.hmb.gov.tr/uploads/2020/05/27_KAF2017021427ihalesonucu10s-1.docx
2025-07-23 23:35:48,712 - INFO - PDF indiriliyor: https://ms.hmb.gov.tr/uploads/2020/05/27_KAF2017021427ihalesonucu10s-1.docx
2025-07-23 23:35:48,795 - ERROR - PDF işlenirken hata: EOF marker not found
2025-07-23 23:35:48,795 - WARNING - ⚠ Bu PDF'den veri çıkarılamadı
2025-07-23 23:35:51,805 - INFO - 
2025-07-23 23:35:51,807 - INFO - İŞLENİYOR (312/315): https://www.hmb.gov.tr/duyuru/7-subat-2017-tarihinde-gerceklestirilen-ihalenin-sonuclarina-iliskin-basin-duyurusu
2025-07-23 23


✓ BAŞARILI! Toplam 398 ihale verisi çekildi

Çekilen ISIN kodları:
1. TRB091220T11 - Hazine Bonosu (2 ihale)
2. TRB110326T12 - Hazine Bonosu (2 ihale)
3. TRB220223T13 - Hazine Bonosu (3 ihale)
4. TRB220420T14 - Hazine Bonosu (1 ihale)
5. TRB230725T15 - Hazine Bonosu (3 ihale)
6. TRT010328T12 - TLREF'e Endeksli Devlet Tahvili (3 ihale)
7. TRT011025T16 - Sabit Kuponlu Devlet Tahvili (15 ihale)
8. TRT020529T18 - TÜFE'ye Endeksli Devlet Tahvili (3 ihale)
9. TRT020926T17 - Sabit Kuponlu Devlet Tahvili (10 ihale)
10. TRT030523T13 - TÜFE'ye Endeksli Devlet Tahvili (1 ihale)
11. TRT031029T10 - Değişken Faizli Devlet Tahvili (8 ihale)
12. TRT040522T13 - Sabit Kuponlu Devlet Tahvili (6 ihale)
13. TRT040729T14 - TLREF'e Endeksli Devlet Tahvili (1 ihale)
14. TRT040832T18 - TÜFE'ye Endeksli Devlet Tahvili (5 ihale)
15. TRT041126T11 - Değişken Faizli Devlet Tahvili (4 ihale)
16. TRT041224T12 - Sabit Kuponlu Devlet Tahvili (6 ihale)
17. TRT050527T17 - Değişken Faizli Devlet Tahvili (6 ihale)
18. TRT

2025-07-23 23:36:17,183 - INFO - ✓ Vade analizi grafikleri vade_analizi.html dosyasına kaydedildi
2025-07-23 23:36:17,183 - INFO - 
2025-07-23 23:36:17,183 - INFO - VADE ANALİZİ ÖZETİ
2025-07-23 23:36:17,183 - INFO - ============================================================
2025-07-23 23:36:17,183 - INFO - Ortalama Vade: 4.99 yıl
2025-07-23 23:36:17,184 - INFO - En Kısa Vade: 2.46 yıl
2025-07-23 23:36:17,184 - INFO - En Uzun Vade: 7.94 yıl
2025-07-23 23:36:17,184 - INFO - Son 3 Ay Ortalaması: 3.77 yıl
2025-07-23 23:36:17,331 - INFO - ✓ Veriler başarıyla hazine_ihale_verileri.xlsx dosyasına kaydedildi.



✓ Vade analizi grafikleri 'vade_analizi.html' dosyasına kaydedildi

✓ Veriler kaydedildi:
  - hazine_ihale_verileri.xlsx
  - hazine_ihale_verileri.csv
  - hazine_vade_analizi.csv
  - vade_analizi.html (interaktif grafikler)
